# SBS Radar — verificación local reproducible

Ejecuta de principio a fin el corpus real de ambas familias y los informes sellados del piloto. **No llama modelos, no inicia recursos ni acredita aceptación E2E.** Ejecutar desde la raíz del snapshot o desde `notebooks/`. La raíz alternativa se configura mediante `SBS_RELEASE_ROOT`; nunca se leen credenciales.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path(os.environ.get('SBS_RELEASE_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'contracts/EvidencePack.json').is_file(), 'Usa el snapshot completo: faltan contratos en la raíz.'
assert (ROOT/'data/retrieval/sk04-real-001/records.json').is_file(), 'Falta el corpus entregable; un wheel o notebook aislado no basta.'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'src'))
print('Raíz validada; configuración local explícita. Python:', sys.version.split()[0])

## Contexto SK00
Se comprueban entradas documentales existentes; no se confunden con permisos operativos.

In [ ]:
from sbs.context import resolve_context
context = resolve_context('SK01-build', root=ROOT)
print(json.dumps({'status': context['status'], 'reusable': context['reusable'], 'gaps': context['gaps']}, indent=2, ensure_ascii=False))
assert context['status'] == 'ready', 'Entradas documentales cambiaron: revisa los hashes antes de continuar.'

## Corpus, citas, contratos e informes sellados
Valida PDFs originales, pasajes contra texto extraído, identidad del índice y ModelBundle. Recalcula métricas guardadas de tres consultas. La referencia IA no es aprobación experta.

In [ ]:
from deployment.offline import verify
result = verify(ROOT)
print(json.dumps(result, ensure_ascii=False, indent=2))
assert result['pdfs'] == 6 and result['records'] == 135
assert len(result['queries']) == 3

## Preflight remoto declarativo
Los recursos observados son candidatos. Un warehouse detenido no se inicia. Los campos no asignados son gates de una futura ejecución remota y no impiden completar esta verificación local.

In [ ]:
from deployment.offline import remote_preflight
preflight = remote_preflight(ROOT)
print(json.dumps(preflight, ensure_ascii=False, indent=2))
assert preflight['resource_actions'] == 0
assert not preflight['ready_for_remote_execution']

## Resultado y siguiente operación

Las celdas anteriores verifican integridad y métricas offline. Para abrir el canal local, consulta `docs/entrega-local.md`; el runtime exige su configuración real. La navegación, inferencia nueva, despliegue autenticado, job diario, permisos de Genie y recuperación siguen gates separados. No hay instalación automática, consultas remotas, creación de recursos ni aprobación institucional en este notebook.